In [ ]:
# ============================================================
# Coupled Euler-Bernoulli Beam + Internal Mass-Spring Chains
# ============================================================
# Packages: LinearAlgebra, DifferentialEquations, Plots
#
# Run once to install:
#   using Pkg
#   Pkg.add(["DifferentialEquations", "Plots"])
#
# DOF layout per beam node: [w, θ]  (transverse, rotation)
# Chain DOFs: relative displacements v^(j) at each attachment node j
# Global DOF vector: q = [w_beam; v^(j1); v^(j2); ...]
# ============================================================

using LinearAlgebra
using DifferentialEquations
using Plots
using Printf

# ============================================================
# 1. EULER-BERNOULLI BEAM ELEMENT MATRICES
# ============================================================
# Hermite cubic shape functions, 2 DOFs/node: [w, θ]
# Element DOF order: [w1, θ1, w2, θ2]

function beam_element_stiffness(E, I, Le)
    c = E * I / Le^3
    return c * [
         12    6Le   -12   6Le;
         6Le   4Le^2 -6Le  2Le^2;
        -12   -6Le   12   -6Le;
         6Le   2Le^2 -6Le  4Le^2
    ]
end

function beam_element_mass(ρ, A, Le)
    c = ρ * A * Le / 420
    return c * [
        156    22Le    54    -13Le;
        22Le   4Le^2   13Le  -3Le^2;
        54     13Le    156   -22Le;
       -13Le  -3Le^2  -22Le   4Le^2
    ]
end

# ============================================================
# 2. ASSEMBLE BEAM GLOBAL MATRICES
# ============================================================
# n_elem elements → n_nodes = n_elem+1 nodes → n_beam_dof = 2*n_nodes DOFs

function assemble_beam(n_elem, E, I, ρ, A, L_total)
    n_nodes   = n_elem + 1
    n_dof     = 2 * n_nodes
    Le        = L_total / n_elem

    Kb = zeros(n_dof, n_dof)
    Mb = zeros(n_dof, n_dof)

    for e in 1:n_elem
        ke   = beam_element_stiffness(E, I, Le)
        me   = beam_element_mass(ρ, A, Le)
        # Global DOF indices for element e (1-indexed)
        dofs = [2*e-1, 2*e, 2*e+1, 2*e+2]
        Kb[dofs, dofs] .+= ke
        Mb[dofs, dofs] .+= me
    end

    return Kb, Mb, n_dof, n_nodes
end

# ============================================================
# 3. CHAIN MATRICES FOR A SINGLE CHAIN
# ============================================================
# masses  : Vector of length n   [m1, m2, ..., mn]
# springs : Vector of length n+1 [k0, k1, ..., kn]
# Returns Mc (n×n diagonal), Kc (n×n tridiagonal) in RELATIVE coords
# BCs v0 = v_{n+1} = 0 are already embedded (walls fixed to beam node)

function chain_matrices(masses::Vector, springs::Vector)
    n = length(masses)
    @assert length(springs) == n + 1 "Need n+1 springs for n masses"

    Mc = Diagonal(masses)

    Kc = zeros(n, n)
    for i in 1:n
        Kc[i, i] += springs[i] + springs[i+1]   # left + right spring
        if i > 1
            Kc[i,   i-1] -= springs[i]           # coupling to left neighbor
            Kc[i-1, i  ] -= springs[i]           # symmetric
        end
    end

    return Matrix(Mc), Kc
end

# ============================================================
# 4. SELECTION VECTOR
# ============================================================
# For beam node j (1-indexed), transverse DOF is at global index 2j-1
# Rotation DOF is at 2j — chains do NOT attach to rotation

function selection_vector(beam_node::Int, n_beam_dof::Int)
    b = zeros(n_beam_dof)
    b[2 * beam_node - 1] = 1.0   # transverse w DOF only
    return b
end

# ============================================================
# 5. GLOBAL ASSEMBLY
# ============================================================
# chains: Vector of NamedTuples (node, Mc, Kc)
# Returns global M and K of size (n_beam_dof + Σn_j) × (n_beam_dof + Σn_j)

function assemble_global(Mb, Kb, chains, n_beam_dof)
    n_chain_dofs = [size(c.Mc, 1) for c in chains]
    n_total      = n_beam_dof + sum(n_chain_dofs)

    M_global = zeros(n_total, n_total)
    K_global = zeros(n_total, n_total)

    # ── Beam block ──────────────────────────────────────────
    M_global[1:n_beam_dof, 1:n_beam_dof] = Mb
    K_global[1:n_beam_dof, 1:n_beam_dof] = Kb

    offset = n_beam_dof
    for c in chains
        n_c   = size(c.Mc, 1)
        b     = selection_vector(c.node, n_beam_dof)    # m-vector
        ones_n = ones(n_c)
        Mc_tot = sum(diag(c.Mc))

        # ── (1,1) beam-beam: add Mc_tot * b bᵀ ─────────────
        M_global[1:n_beam_dof, 1:n_beam_dof] .+= Mc_tot * (b * b')

        # ── (1,j) beam-chain: b 1ᵀ Mc ──────────────────────
        chain_range = offset+1 : offset+n_c
        M_global[1:n_beam_dof, chain_range] .+= b * (ones_n' * c.Mc)

        # ── (j,1) chain-beam: Mc 1 bᵀ  (symmetric) ─────────
        M_global[chain_range, 1:n_beam_dof] .+= c.Mc * ones_n * b'

        # ── (j,j) chain-chain diagonal blocks ───────────────
        M_global[chain_range, chain_range]   = c.Mc
        K_global[chain_range, chain_range]   = c.Kc

        offset += n_c
    end

    return M_global, K_global
end

# ============================================================
# 6. BOUNDARY CONDITIONS
# ============================================================
# Simply supported: fix w (not θ) at nodes 1 and n_nodes
# Clamped-clamped : fix w and θ at nodes 1 and n_nodes
# Returns free DOF indices only

function simply_supported_dofs(n_nodes, n_beam_dof)
    # Fix transverse DOF at node 1 (idx 1) and node n_nodes (idx 2n_nodes-1)
    fixed = [1, 2*n_nodes - 1]
    return setdiff(1:n_beam_dof, fixed)
end

function clamped_clamped_dofs(n_nodes, n_beam_dof)
    # Fix w and θ at both ends
    fixed = [1, 2, 2*n_nodes-1, 2*n_nodes]
    return setdiff(1:n_beam_dof, fixed)
end

function fixed_free_dofs(n_nodes, n_beam_dof)
    # Fix w and θ at node 1 only (indices 1 and 2)
    fixed = [1, 2]
    return setdiff(1:n_beam_dof, fixed)
end

function apply_bcs(M_global, K_global, free_beam_dofs, n_beam_dof, n_total)
    # Chain DOFs are never constrained — always free
    chain_dofs  = n_beam_dof+1 : n_total
    free_global = vcat(free_beam_dofs, collect(chain_dofs))
    return M_global[free_global, free_global],
           K_global[free_global, free_global],
           free_global
end

# ============================================================
# 7. EIGENVALUE SOLVE  (natural frequencies + mode shapes)
# ============================================================
# Solves generalized problem: K φ = ω² M φ

function solve_modes(M_free, K_free, n_modes::Int)
    vals, vecs = eigen(Symmetric(K_free), Symmetric(M_free))
    # Keep only positive eigenvalues (discard numerical negatives near 0)
    pos    = findall(vals .> 1e-8)
    vals   = vals[pos]
    vecs   = vecs[:, pos]
    n_keep = min(n_modes, length(vals))
    ωn     = sqrt.(vals[1:n_keep])          # rad/s
    fn     = ωn ./ (2π)                     # Hz
    Φ      = vecs[:, 1:n_keep]              # mode shapes (columns)
    return ωn, fn, Φ
end

# =================================================================
# 8. TIME INTEGRATION  (second-order ODE via DifferentialEquations)
# =================================================================
# Equation: M q̈ + K q = F(t)
# Recast as first-order: ẏ = [q̇; -M⁻¹(Kq - F)]

function time_integrate(M_free, K_free, F_free_func, q0, qdot0, tspan)
    # Pre-factorise M for speed
    M_chol = cholesky(Symmetric(M_free))

    function ode!(dY, Y, p, t)
        n    = length(q0)
        q    = Y[1:n]
        qdot = Y[n+1:end]
        F    = F_free_func(t)              # external force vector (free DOFs)
        dY[1:n]     = qdot
        dY[n+1:end] = M_chol \ (F - K_free * q)
    end

    Y0      = vcat(q0, qdot0)
    problem = ODEProblem(ode!, Y0, tspan)
    sol     = solve(problem, Tsit5(); reltol=1e-8, abstol=1e-10)
    return sol
end

# ============================================================
# 9. EXAMPLE PROBLEM
# ============================================================
# Simply-supported steel beam, 10 nodes (9 elements), 1 m span
# Chain at node 3: 10 masses, 11 springs
# Chain at node 5:  5 masses,  6 springs

# function run_example()

#     # ── Beam properties ─────────────────────────────────────
#     E       = 210e9      # Pa  (steel)
#     ρ       = 7800.0     # kg/m³
#     b_width = 0.05       # m   cross-section width
#     h_depth = 0.05       # m   cross-section height
#     A       = b_width * h_depth
#     I       = b_width * h_depth^3 / 12
#     L       = 1.0        # m   total beam length
#     n_elem  = 9
#     n_nodes = n_elem + 1   # = 10

#     Kb, Mb, n_beam_dof, _ = assemble_beam(n_elem, E, I, ρ, A, L)

#     # ── Chain at node 3: 10 masses, 11 springs ───────────────
#     m3  = fill(0.05, 10)              # kg each
#     k3  = fill(500.0, 11)            # N/m each
#     Mc3, Kc3 = chain_matrices(m3, k3)

#     # ── Chain at node 5:  5 masses,  6 springs ───────────────
#     m5  = fill(0.05, 5)
#     k5  = fill(800.0, 6)
#     Mc5, Kc5 = chain_matrices(m5, k5)

#     chains = [
#         (node=3, Mc=Mc3, Kc=Kc3),
#         (node=5, Mc=Mc5, Kc=Kc5),
#     ]

#     # ── Global assembly ──────────────────────────────────────
#     n_total            = n_beam_dof + 10 + 5
#     M_global, K_global = assemble_global(Mb, Kb, chains, n_beam_dof)

#     # ── Boundary conditions (simply supported) ───────────────
#     free_beam  = simply_supported_dofs(n_nodes, n_beam_dof)
#     M_free, K_free, free_global = apply_bcs(
#         M_global, K_global, free_beam, n_beam_dof, n_total)

#     println("Total DOFs (before BCs): ", n_total)
#     println("Free DOFs  (after  BCs): ", length(free_global))

#     # ── Natural frequencies ──────────────────────────────────
#     n_modes       = 12
#     ωn, fn, Φ    = solve_modes(M_free, K_free, n_modes)

#     println("\nFirst $n_modes natural frequencies:")
#     for i in 1:n_modes
#         @printf("  Mode %2d:  %8.3f Hz  (%8.3f rad/s)\n", i, fn[i], ωn[i])
#     end

#     # ── Time integration ─────────────────────────────────────
#     # Apply a half-sine impulse at mid-span node (node 5 → transverse DOF)
#     # Find the free-DOF index corresponding to beam node 5 transverse DOF
#     beam_node5_global = 2*5 - 1               # global beam DOF (transverse)
#     local_idx = findfirst(==(beam_node5_global), free_global)

#     F0    = 100.0    # N   peak force
#     t_imp = 0.005    # s   impulse half-period

#     function F_free(t)
#         F = zeros(length(free_global))
#         if !isnothing(local_idx) && t <= t_imp
#             F[local_idx] = F0 * sin(π * t / t_imp)
#         end
#         return F
#     end

#     q0    = zeros(length(free_global))
#     qdot0 = zeros(length(free_global))
#     tspan = (0.0, 0.5)

#     println("\nRunning time integration...")
#     sol = time_integrate(M_free, K_free, F_free, q0, qdot0, tspan)
#     println("Done. $(length(sol.t)) time steps.")

#     # ── Reconstruct full response at beam nodes ───────────────
#     # Extract transverse DOFs of beam nodes 1..n_nodes from free solution
#     t_plot   = range(tspan[1], tspan[2], length=500)
#     q_full   = hcat([sol(t) for t in t_plot]...)  # (n_free) × (n_t)

#     # Map free transverse beam DOFs back
#     beam_w_free_indices = Int[]    # indices in free_global that are beam w DOFs
#     beam_node_labels    = Int[]
#     for node in 1:n_nodes
#         gdof = 2*node - 1          # global transverse DOF
#         idx  = findfirst(==(gdof), free_global)
#         if !isnothing(idx)
#             push!(beam_w_free_indices, idx)
#             push!(beam_node_labels,   node)
#         end
#     end

#     w_beam = q_full[beam_w_free_indices, :]   # (n_free_nodes) × (n_t)

#     # Chain 1 (node 3) relative DOFs — first 10 after beam DOFs
#     chain1_start = length(free_beam) + 1
#     chain1_end   = chain1_start + 9
#     v_chain1     = q_full[chain1_start:chain1_end, :]

#     # Absolute displacement of chain 1 masses = w_{j=3} + v_i
#     node3_local  = findfirst(==(3), beam_node_labels)
#     w_node3      = isnothing(node3_local) ? zeros(1, size(w_beam,2)) :
#                    w_beam[node3_local:node3_local, :]
#     u_chain1     = w_node3 .+ v_chain1

#     # # ── Plotting ─────────────────────────────────────────────
#     # t_vec = collect(t_plot)

#     # # Plot 1: Beam transverse response at selected nodes
#     # p1 = plot(t_vec, w_beam' .* 1e3;
#     #     label  = ["Node $(n)" for n in beam_node_labels],
#     #     xlabel = "Time (s)",
#     #     ylabel = "Transverse displacement w (mm)",
#     #     title  = "Beam node transverse response",
#     #     lw     = 1.5,
#     #     legend = :topright)

#     # # Plot 2: Natural frequency spectrum
#     # p2 = bar(1:n_modes, fn;
#     #     xlabel = "Mode number",
#     #     ylabel = "Natural frequency (Hz)",
#     #     title  = "Natural frequencies",
#     #     legend = false,
#     #     color  = :steelblue)

#     # # Plot 3: Chain 1 absolute displacements (all 10 masses)
#     # p3 = plot(t_vec, u_chain1' .* 1e3;
#     #     label  = ["Mass $i" for i in 1:10],
#     #     xlabel = "Time (s)",
#     #     ylabel = "Absolute displacement (mm)",
#     #     title  = "Chain at node 3 — mass displacements",
#     #     lw     = 1.2,
#     #     legend = :outerright)

#     # # Plot 4: Mode shape 1 (beam transverse DOFs only)
#     # mode1_full  = zeros(n_total)
#     # mode1_full[free_global] = Φ[:, 1]
#     # w_mode1     = [mode1_full[2*n - 1] for n in 1:n_nodes]
#     # node_coords = range(0, L, length=n_nodes)

#     # p4 = plot(node_coords, w_mode1;
#     #     marker = :circle,
#     #     xlabel = "x (m)",
#     #     ylabel = "Normalized displacement",
#     #     title  = "Mode shape 1  (f₁ = $(round(fn[1], digits=2)) Hz)",
#     #     legend = false,
#     #     lw     = 2.0)

#     # # Combine and save
#     # fig = plot(p1, p2, p3, p4; layout=(2,2), size=(1200, 800), margin=5Plots.mm)
#     # savefig(fig, "coupled_beam_chain_results.png")
#     # println("\nFigure saved → coupled_beam_chain_results.png")

#     # ── Prepare mode shape data ───────────────────────────────
#     mode1_full  = zeros(n_total)
#     mode1_full[free_global] = Φ[:, 1]
#     w_mode1     = [mode1_full[2*n - 1] for n in 1:n_nodes]
#     node_coords = range(0, L, length=n_nodes)

#     # ── Plotting (corrected layout) ─────────────────────────────
#     t_vec = collect(t_plot)

#     # Use explicit layout to control proportions
#     l = @layout [
#         a{0.5w} b{0.5w}
#         c{0.5w} d{0.5w}
#     ]

#     # Choose which nodes to plot
#     nodes_to_plot = [2]

#     # Find corresponding row indices in w_beam
#     rows = [findfirst(==(n), beam_node_labels) for n in nodes_to_plot]

#     # Plot 1: Beam transverse response
#     p1 = plot(t_vec, permutedims(w_beam[rows, :]) .* 1e3;
#         label = reshape(["Node $n" for n in nodes_to_plot], 1, :),
#         xlabel = "Time (s)",
#         ylabel = "Transverse displacement (mm)",
#         title  = "Beam node response",
#         lw     = 1.5,
#         legend = :topright,
#         legendfontsize = 6,
#         margin = 3Plots.mm)

#     # Plot 2: Natural frequency spectrum
#     p2 = bar(1:n_modes, fn;
#         xlabel = "Mode number",
#         ylabel = "Natural frequency (Hz)",
#         title  = "Natural frequencies",
#         legend = false,
#         color  = :steelblue,
#         margin = 3Plots.mm)

#     # Plot 3: Chain absolute displacements (masses 1,3,5,7,9 only for clarity)
#     # Showing all 10 masses makes the legend huge; pick a subset
#     chain_masses_to_show = [1,3]
#     p3 = plot(t_vec, permutedims(u_chain1[chain_masses_to_show, :]) .* 1e3;
#         label = reshape(["Mass $i" for i in chain_masses_to_show], 1, :),
#         xlabel = "Time (s)",
#         ylabel = "Absolute displacement (mm)",
#         title  = "Chain at node 3 — selected masses",
#         lw     = 1.2,
#         legend = :topright,
#         legendfontsize = 6,
#         margin = 3Plots.mm)

#     # Plot 4: First mode shape
#     p4 = plot(node_coords, w_mode1;
#         marker = :circle,
#         xlabel = "x (m)",
#         ylabel = "Normalized displacement",
#         title  = "Mode 1 (f₁ = $(round(fn[1], digits=2)) Hz)",
#         legend = false,
#         lw     = 2.0,
#         margin = 3Plots.mm)

#     # Combine with explicit layout and size
#     fig = plot(p1, p2, p3, p4; layout=l, size=(1200, 800))

#     # Display interactively (optional)
#     display(fig)

#     # Short pause to let GR backend settle (sometimes helps)
#     sleep(0.5)

#     # Save figure
#     savefig(fig, "coupled_beam_chain_results.png")
#     println("\nFigure saved → coupled_beam_chain_results.png")

#     return M_global, K_global, ωn, fn, Φ, sol
# end

# # ============================================================
# # 10. ENTRY POINT
# # ============================================================

# M_global, K_global, ωn, fn, Φ, sol = run_example()

# ============================================================
# EXAMPLE CASE: CANTILEVER BEAM WITH INTERNAL CHAINS
# ============================================================
function run_example_cantilever()
    # ── Beam properties (same as before) ─────────────────────
    E       = 210e9      # Pa  (steel)
    ρ       = 7800.0     # kg/m³
    b_width = 0.05       # m   cross-section width
    h_depth = 0.05       # m   cross-section height
    A       = b_width * h_depth
    I       = b_width * h_depth^3 / 12
    L       = 1.0        # m   total beam length
    n_elem  = 9
    n_nodes = n_elem + 1   # = 10

    Kb, Mb, n_beam_dof, _ = assemble_beam(n_elem, E, I, ρ, A, L)

    # ── Chain at node 3: 10 masses, 11 springs ───────────────
    m3  = fill(0.05, 10)              # kg each
    k3  = fill(500.0, 11)            # N/m each
    Mc3, Kc3 = chain_matrices(m3, k3)

    # ── Chain at node 5:  5 masses,  6 springs ───────────────
    m5  = fill(0.05, 5)
    k5  = fill(800.0, 6)
    Mc5, Kc5 = chain_matrices(m5, k5)

    chains = [
        (node=3, Mc=Mc3, Kc=Kc3),
        (node=5, Mc=Mc5, Kc=Kc5),
    ]

    # ── Global assembly ──────────────────────────────────────
    n_total            = n_beam_dof + 10 + 5
    M_global, K_global = assemble_global(Mb, Kb, chains, n_beam_dof)

    # ── Boundary conditions: FIXED-FREE (cantilever) ─────────
    free_beam  = fixed_free_dofs(n_nodes, n_beam_dof)
    M_free, K_free, free_global = apply_bcs(
        M_global, K_global, free_beam, n_beam_dof, n_total)

    println("Total DOFs (before BCs): ", n_total)
    println("Free DOFs  (after  BCs): ", length(free_global))

    # ── Natural frequencies ──────────────────────────────────
    n_modes = 12
    ωn, fn, Φ = solve_modes(M_free, K_free, n_modes)

    println("\nFirst $n_modes natural frequencies (cantilever):")
    for i in 1:n_modes
        @printf("  Mode %2d:  %8.3f Hz  (%8.3f rad/s)\n", i, fn[i], ωn[i])
    end

    # ── Time integration ─────────────────────────────────────
    # Impulse at free end (node 10) transverse DOF
    beam_node10_global = 2*10 - 1               # global beam DOF (transverse)
    local_idx = findfirst(==(beam_node10_global), free_global)

    F0    = 100.0    # N   peak force
    t_imp = 0.005    # s   impulse half-period

    function F_free(t)
        F = zeros(length(free_global))
        if !isnothing(local_idx) && t <= t_imp
            F[local_idx] = F0 * sin(π * t / t_imp)
        end
        return F
    end

    q0    = zeros(length(free_global))
    qdot0 = zeros(length(free_global))
    tspan = (0.0, 0.5)

    println("\nRunning time integration...")
    sol = time_integrate(M_free, K_free, F_free, q0, qdot0, tspan)
    println("Done. $(length(sol.t)) time steps.")

    # ── Reconstruct response (similar to original) ───────────
    t_plot   = range(tspan[1], tspan[2], length=500)
    q_full   = hcat([sol(t) for t in t_plot]...)

    beam_w_free_indices = Int[]
    beam_node_labels    = Int[]
    for node in 1:n_nodes
        gdof = 2*node - 1
        idx  = findfirst(==(gdof), free_global)
        if !isnothing(idx)
            push!(beam_w_free_indices, idx)
            push!(beam_node_labels,   node)
        end
    end

    w_beam = q_full[beam_w_free_indices, :]

    chain1_start = length(free_beam) + 1
    chain1_end   = chain1_start + 9
    v_chain1     = q_full[chain1_start:chain1_end, :]

    node3_local  = findfirst(==(3), beam_node_labels)
    w_node3      = isnothing(node3_local) ? zeros(1, size(w_beam,2)) :
                   w_beam[node3_local:node3_local, :]
    u_chain1     = w_node3 .+ v_chain1

    # ── Plotting (simplified) ────────────────────────────────
    t_vec = collect(t_plot)
    nodes_to_plot = [2, 5, 10]
    rows = [findfirst(==(n), beam_node_labels) for n in nodes_to_plot]

    p1 = plot(t_vec, permutedims(w_beam[rows, :]) .* 1e3;
        label = reshape(["Node $n" for n in nodes_to_plot], 1, :),
        xlabel = "Time (s)", ylabel = "w (mm)",
        title = "Cantilever beam response",
        lw = 1.5, legend = :topright)

    p2 = bar(1:n_modes, fn;
        xlabel = "Mode number", ylabel = "Frequency (Hz)",
        title = "Natural frequencies", legend = false, color = :steelblue)

    p3 = plot(t_vec, permutedims(u_chain1[[1,3,5], :]) .* 1e3;
        label = reshape(["Mass $i" for i in [1,3,5]], 1, :),
        xlabel = "Time (s)", ylabel = "Absolute displacement (mm)",
        title = "Chain at node 3 — selected masses", lw = 1.2, legend = :topright)

    mode1_full = zeros(n_total)
    mode1_full[free_global] = Φ[:, 1]
    w_mode1 = [mode1_full[2*n - 1] for n in 1:n_nodes]
    node_coords = range(0, L, length=n_nodes)

    p4 = plot(node_coords, w_mode1;
        marker = :circle, xlabel = "x (m)", ylabel = "Normalized displacement",
        title = "Mode 1 (f₁ = $(round(fn[1], digits=2)) Hz)",
        legend = false, lw = 2.0)

    fig = plot(p1, p2, p3, p4; layout=(2,2), size=(1200, 800))
    display(fig)
    savefig(fig, "coupled_beam_chain_cantilever_results.png")
    println("\nFigure saved → coupled_beam_chain_cantilever_results.png")

    return M_global, K_global, ωn, fn, Φ, sol
end

run_example_cantilever()

Total DOFs (before BCs): 